# Text Generation Question–Answer System with FLAN-T5

This notebook demonstrates the same workflow as `app.py`: question → instruction prompt → tokenization → FLAN-T5 → generated tokens → decoded answer → Gradio. The model uses learned knowledge and may be wrong; it does not answer from a supplied document collection. First use downloads packages and model files and requires internet access and sufficient memory.

In [ ]:
%pip install -q "transformers>=4.45,<5" "sentencepiece>=0.2,<1" "accelerate>=1.2,<2" "gradio>=5,<6"

In [ ]:
import logging
import torch
import gradio as gr
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

logging.basicConfig(level=logging.INFO)
MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 128
NUM_BEAMS = 4
DO_SAMPLE = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Selected device: {device}")

In [ ]:
# Load once here, not again for each submitted question.
print(f"Loading {MODEL_NAME} from Hugging Face...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
print(f"Ready: {MODEL_NAME} on {device}")

## Generate an answer

The function validates input, builds an instruction, tokenizes it, runs inference, and decodes the resulting token IDs. Exceptions are logged and returned as an explanatory message.

In [ ]:
def generate_answer(question):
    if not isinstance(question, str) or not question.strip():
        return "Please enter a question before submitting."

    prompt = (
        "Answer the following question clearly and concisely. "
        "If you are unsure, say so.\n"
        f"Question: {question.strip()}\nAnswer:"
    )
    try:
        # Tokenize the instruction prompt into model-readable tensors.
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        )
        inputs = {key: value.to(device) for key, value in inputs.items()}

        # Generate token IDs without calculating training gradients.
        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                num_beams=NUM_BEAMS,
                do_sample=DO_SAMPLE,
            )

        answer = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()
        return answer or "The model returned an empty answer. Please try rephrasing."
    except Exception as exc:
        logging.exception("Answer generation failed")
        return f"Answer generation failed ({type(exc).__name__}: {exc})"

## Sample response analysis

Run this cell to generate and display actual responses. For your practical report, record the output shown by your own run; wording can vary. Do not treat example questions or anticipated answers as generated results.

In [ ]:
test_questions = [
    "What is Artificial Intelligence?",
    "Explain how a Transformer works.",
    "What is the difference between AI and Machine Learning?",
]
for question in test_questions:
    print(f"Question: {question}")
    print(f"Generated response: {generate_answer(question)}\n")

## Gradio interface

The default launch is private to the current notebook runtime. If you explicitly enable `share=True`, the generated link may be public and temporary. Do not enter private or sensitive information.

In [ ]:
with gr.Blocks(title="Text Generation Question–Answer System") as demo:
    gr.Markdown(
        "# Text Generation Using FLAN-T5\n"
        "Ask a question and generate an answer with a pre-trained Transformer."
    )
    question_input = gr.Textbox(
        label="Your question",
        placeholder="For example: What is Artificial Intelligence?",
        lines=3,
    )
    with gr.Row():
        submit_button = gr.Button("Generate answer", variant="primary")
        clear_button = gr.Button("Clear")
    answer_output = gr.Textbox(label="Generated answer", lines=6, interactive=False)
    gr.Examples(
        examples=[
            ["What is Artificial Intelligence?"],
            ["Explain how a Transformer works."],
            ["What is Generative AI?"],
            ["What is the difference between AI and Machine Learning?"],
            ["Explain neural networks in simple words."],
        ],
        inputs=question_input,
    )
    gr.Markdown(
        f"**Model:** `{MODEL_NAME}`  \n"
        f"**Device:** `{device}`  \n"
        "**Workflow:** question → prompt → tokens → model → generated tokens → answer.  "
        "Responses may be incorrect or incomplete; verify important information."
    )
    submit_button.click(
        generate_answer, inputs=question_input, outputs=answer_output, show_progress="full"
    )
    question_input.submit(
        generate_answer, inputs=question_input, outputs=answer_output, show_progress="full"
    )
    clear_button.click(lambda: ("", ""), outputs=[question_input, answer_output])
# Set share=True only if you need a temporary public link and understand the risk.
demo.queue().launch(share=False)